# Fusion Head — DefakeX Combined System

## Architecture
```
freq_logit  ──┐
               ├──► FusionMLP(2→32→16→1) ──► final_logit ──► real/fake
spatial_logit ──┘
```

## What the fusion head learns
- When freq_high + spatial_low → trust freq (AI-generated image)
- When freq_low + spatial_high → trust spatial (deepfake manipulation)
- When freq_high + spatial_low + phone characteristics → output real
- When both agree → amplify confidence

## Training data sources
- StyleGAN + Flickr + DeepDetect + OpenFake (from frequency branch training)
- FF++ all manipulation types (from spatial branch training)
- Phone images 800 train / 521 test (the critical conflict case)
- CelebDF: held-out test only

## Both branch models are FROZEN — only fusion MLP is trained

In [ ]:
import os, io, gc, random, csv
from pathlib import Path
from dataclasses import dataclass
from collections import Counter, defaultdict

import numpy as np
from PIL import Image, ImageFilter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b3
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_curve, auc
)
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

In [ ]:
# ══════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════
SEED       = 42
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

# Fusion head training
EPOCHS     = 20
BATCH_SIZE = 256   # large batch — tiny network, fast training
LR         = 3e-4
LR_MIN     = 1e-6
WEIGHT_DECAY = 1e-4

# Image preprocessing (must match each branch exactly)
TARGET_SIZE   = 224
RESIZE_SIZE   = 256
FFT_CLIP_MAX  = 14.0
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_EXTS      = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

# Phone image split
PHONE_TRAIN_N = 800   # used in fusion training
PHONE_TEST_N  = 521   # held-out test

# Dataset caps (same as frequency branch training)
CAPS = {
    'stylegan_real':   50000,
    'stylegan_fake':   50000,
    'flickr_real':     30000,
    'deepdetect_fake': 30000,
}
OPENFAKE_MANIFEST = '/kaggle/input/datasets/frequency-model-checkpoint/of_manifest.csv'

# Checkpoints
FREQ_CKPT    = '/kaggle/input/datasets/frequency-model-checkpoint/best_model.pth'
SPATIAL_CKPT = '/kaggle/input/datasets/spatial-v2-checkpoint/best_model_spatial_v2.pth'
CKPT_DIR     = '/kaggle/working'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('Device:', DEVICE)
print('Config OK')

In [ ]:
# ══════════════════════════════════════════════
#  DATASET PATHS
# ══════════════════════════════════════════════

# Frequency branch datasets (real vs AI-generated)
STYLEGAN_ROOT   = Path('/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train')
FLICKR_ROOT     = Path('/kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images')
DEEPDETECT_ROOT = Path('/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train')

# Spatial branch dataset (deepfake manipulations)
FF_SPLIT_ROOT = Path('/kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k/dataset_processed_split')
FF_FAKE_TYPES = ['Deepfakes','Face2Face','FaceShifter','FaceSwap','NeuralTextures','DeepFakeDetection']

# Phone images — real photos, the critical conflict case
PHONE_ROOT = Path('/kaggle/input/datasets/kashirhanif/phone-images')

# CelebDF — held-out cross-dataset test only
CELEB_ROOT = Path('/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2')

for name, p in [
    ('StyleGAN',    STYLEGAN_ROOT),
    ('Flickr',      FLICKR_ROOT),
    ('DeepDetect',  DEEPDETECT_ROOT),
    ('FF++ splits', FF_SPLIT_ROOT),
    ('Phone images',PHONE_ROOT),
    ('CelebDF',     CELEB_ROOT),
    ('OpenFake manifest', Path(OPENFAKE_MANIFEST)),
    ('Freq checkpoint',   Path(FREQ_CKPT)),
    ('Spatial checkpoint',Path(SPATIAL_CKPT)),
]:
    status = 'OK' if Path(p).exists() else 'MISSING'
    print(f'  {name:22s}: {status}')

In [ ]:
# ══════════════════════════════════════════════
#  PREPROCESSING — must match branch training exactly
# ══════════════════════════════════════════════

def preprocess_frequency(img: Image.Image) -> torch.Tensor:
    """Identical to frequency branch training pipeline."""
    img  = img.convert('RGB')
    img  = img.resize((RESIZE_SIZE, RESIZE_SIZE), Image.BICUBIC)
    left = (RESIZE_SIZE - TARGET_SIZE) // 2
    img  = img.crop((left, left, left + TARGET_SIZE, left + TARGET_SIZE))
    arr  = np.array(img, dtype=np.float32) / 255.0
    x    = torch.from_numpy(arr).permute(2, 0, 1)
    out  = []
    for c in range(3):
        f   = torch.fft.fftshift(torch.fft.fft2(x[c]))
        mag = torch.log1p(torch.abs(f))
        mag = torch.clamp(mag, 0.0, FFT_CLIP_MAX) / FFT_CLIP_MAX
        out.append(mag)
    return torch.stack(out)


def preprocess_spatial(img: Image.Image) -> torch.Tensor:
    """Identical to spatial branch training pipeline."""
    img  = img.convert('RGB')
    img  = img.resize((RESIZE_SIZE, RESIZE_SIZE), Image.BICUBIC)
    left = (RESIZE_SIZE - TARGET_SIZE) // 2
    img  = img.crop((left, left, left + TARGET_SIZE, left + TARGET_SIZE))
    arr  = np.array(img, dtype=np.float32) / 255.0
    mean = np.array(IMAGENET_MEAN, dtype=np.float32)
    std  = np.array(IMAGENET_STD,  dtype=np.float32)
    arr  = (arr - mean) / std
    return torch.from_numpy(arr).permute(2, 0, 1).float()


print('Preprocessing functions defined.')

In [ ]:
# ══════════════════════════════════════════════
#  LOAD BOTH BRANCH MODELS — FROZEN
#  Both branches are fixed during fusion training.
#  Only the fusion MLP weights are updated.
# ══════════════════════════════════════════════
def build_efficientnet():
    m = efficientnet_b3(weights=None)
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(m.classifier[1].in_features, 1)
    )
    return m


# Frequency branch
freq_model = build_efficientnet().to(DEVICE)
freq_ckpt  = torch.load(FREQ_CKPT, map_location=DEVICE)
freq_model.load_state_dict(freq_ckpt['model_state_dict'])
freq_model.eval()
for p in freq_model.parameters(): p.requires_grad = False
print(f'Frequency branch loaded  — epoch {freq_ckpt.get("epoch","?")}  '
      f'(frozen, {sum(p.numel() for p in freq_model.parameters())/1e6:.1f}M params)')

# Spatial branch v2
spatial_model = build_efficientnet().to(DEVICE)
sp_ckpt = torch.load(SPATIAL_CKPT, map_location=DEVICE)
spatial_model.load_state_dict(sp_ckpt['model_state_dict'])
spatial_model.eval()
for p in spatial_model.parameters(): p.requires_grad = False
print(f'Spatial branch v2 loaded — epoch {sp_ckpt.get("epoch","?")}  '
      f'(frozen, {sum(p.numel() for p in spatial_model.parameters())/1e6:.1f}M params)')

In [ ]:
# ══════════════════════════════════════════════
#  FUSION HEAD
#
#  Input:  2 raw logits [freq_logit, spatial_logit]
#  Output: 1 final logit → sigmoid → real/fake probability
#
#  Why raw logits not probabilities:
#  Logits preserve magnitude information (how confident each branch is).
#  sigmoid(5.0) and sigmoid(1.0) both look 'high' as probabilities
#  but the fusion head can distinguish extreme confidence from moderate.
#
#  Architecture: 2→32→16→1 with BatchNorm and ReLU
#  Small enough to train in a few epochs on logit pairs.
#  Large enough to learn non-linear combinations.
# ══════════════════════════════════════════════
class FusionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(16, 1),
        )

    def forward(self, freq_logit, spatial_logit):
        x = torch.stack([freq_logit, spatial_logit], dim=1)  # (B, 2)
        return self.net(x).squeeze(1)                         # (B,)


fusion = FusionHead().to(DEVICE)
n_params = sum(p.numel() for p in fusion.parameters())
print(f'FusionHead: {n_params:,} trainable parameters')
print('Architecture:', fusion)

In [ ]:
@dataclass(frozen=True)
class SampleRef:
    path:   str
    label:  int    # 0=real  1=fake
    source: str

def list_images(p: Path):
    if not p.exists(): return []
    return [x for x in p.rglob('*')
            if x.is_file() and x.suffix.lower() in IMG_EXTS]

def cap_shuffle(paths, cap):
    random.shuffle(paths)
    return paths[:cap] if cap and len(paths) > cap else paths

def split_paths(paths, label, source, tr=0.75, va=0.10):
    random.shuffle(paths)
    n = len(paths); nt = int(tr*n); nv = int(va*n)
    mk = lambda ps: [SampleRef(str(p), label, source) for p in ps]
    return mk(paths[:nt]), mk(paths[nt:nt+nv]), mk(paths[nt+nv:])

def split_refs_per_source(refs, tr=0.75, va=0.10):
    by_src = defaultdict(list)
    for r in refs: by_src[r.source].append(r)
    tr_out, va_out, te_out = [], [], []
    for src_refs in by_src.values():
        random.shuffle(src_refs)
        n = len(src_refs); nt = int(tr*n); nv = int(va*n)
        tr_out += src_refs[:nt]
        va_out += src_refs[nt:nt+nv]
        te_out += src_refs[nt+nv:]
    return tr_out, va_out, te_out

print('SampleRef helpers defined.')

In [ ]:
# ══════════════════════════════════════════════
#  BUILD FUSION TRAINING SET
#
#  Four scenario types the fusion head must learn:
#
#  1. AI-generated fakes (OpenFake + StyleGAN fake + DeepDetect)
#     → freq HIGH, spatial LOW → output FAKE
#     → teaches: trust frequency branch for AI images
#
#  2. Deepfake manipulations (FF++ all types)
#     → freq LOW, spatial HIGH → output FAKE
#     → teaches: trust spatial branch for face swaps
#
#  3. Clean real faces (StyleGAN real + Flickr + OpenFake real)
#     → freq LOW, spatial LOW → output REAL
#     → teaches: both branches agreeing real = real
#
#  4. Phone real photos (critical conflict case)
#     → freq HIGH, spatial LOW → output REAL
#     → teaches: override frequency branch for phone images
#     → Without this, fusion inherits frequency branch phone bias
# ══════════════════════════════════════════════
print('Building fusion training set...')
train_refs, val_refs, test_refs = [], [], []

# ── Scenario 1+3: Frequency branch datasets ──
# (StyleGAN real/fake + Flickr real + DeepDetect fake + OpenFake)
sty_real = cap_shuffle(list_images(STYLEGAN_ROOT / 'real'), CAPS['stylegan_real'])
sty_fake = cap_shuffle(list_images(STYLEGAN_ROOT / 'fake'), CAPS['stylegan_fake'])
flk_real = cap_shuffle(list_images(FLICKR_ROOT),            CAPS['flickr_real'])
dd_fake  = cap_shuffle(list_images(DEEPDETECT_ROOT / 'fake'), CAPS['deepdetect_fake'])

print(f'  StyleGAN real: {len(sty_real):,}')
print(f'  StyleGAN fake: {len(sty_fake):,}')
print(f'  Flickr real:   {len(flk_real):,}')
print(f'  DeepDetect fk: {len(dd_fake):,}')

for paths, lbl, src in [
    (sty_real, 0, 'stylegan_real'),
    (sty_fake, 1, 'stylegan_fake'),
    (flk_real, 0, 'flickr_real'),
    (dd_fake,  1, 'deepdetect_fake'),
]:
    tr, va, te = split_paths(paths, lbl, src)
    train_refs += tr; val_refs += va; test_refs += te

# OpenFake (AI-generated, 33 generators)
if os.path.exists(OPENFAKE_MANIFEST):
    of_refs = []
    with open(OPENFAKE_MANIFEST, 'r') as f:
        for row in csv.DictReader(f):
            g   = row['source']
            src = 'openfake_real' if g == 'real' else f'openfake_{g}'
            of_refs.append(SampleRef(row['path'], int(row['label']), src))
    of_tr, of_va, of_te = split_refs_per_source(of_refs)
    train_refs += of_tr; val_refs += of_va; test_refs += of_te
    print(f'  OpenFake:      {len(of_refs):,}')
else:
    print('  OpenFake manifest not found — skipping')

# ── Scenario 2: FF++ deepfake manipulations ──
print('\nLoading FF++ splits...')
for split_name, ref_list in [('train',train_refs),('val',val_refs),('test',test_refs)]:
    sd = FF_SPLIT_ROOT / split_name
    if not sd.exists(): continue
    # FF++ Real already covered by StyleGAN+Flickr — only add fakes
    for ft in FF_FAKE_TYPES:
        fd = sd / ft
        if not fd.exists(): continue
        fp = cap_shuffle(list_images(fd), None)
        ref_list += [SampleRef(str(p), 1, f'ff_{ft}') for p in fp]

print(f'  FF++ fakes added to all splits')

# ── Scenario 4: Phone real images ──
# Split: 800 train, 200 val, 521 test (held-out)
# These are the critical conflict case:
# frequency says fake (mean_prob=0.72), spatial says real (mean_prob=0.26)
# Fusion head must learn: override frequency, output real
phone_all = cap_shuffle(list_images(PHONE_ROOT), None)
print(f'\n  Phone images found: {len(phone_all):,}')
random.shuffle(phone_all)
phone_train = phone_all[:PHONE_TRAIN_N]
phone_val   = phone_all[PHONE_TRAIN_N:PHONE_TRAIN_N+200]
phone_test  = phone_all[PHONE_TRAIN_N+200:]
train_refs += [SampleRef(str(p), 0, 'phone_real') for p in phone_train]
val_refs   += [SampleRef(str(p), 0, 'phone_real') for p in phone_val]
phone_test_refs = [SampleRef(str(p), 0, 'phone_real') for p in phone_test]
print(f'  Phone split: train={len(phone_train)} val={len(phone_val)} test={len(phone_test)}')

random.shuffle(train_refs); random.shuffle(val_refs); random.shuffle(test_refs)

print('\nFinal fusion training set:')
for name, refs in [('Train',train_refs),('Val',val_refs),('Test',test_refs)]:
    lc = Counter(r.label for r in refs)
    sc = Counter(r.source for r in refs)
    print(f'  {name}: {len(refs):,}  real={lc[0]:,}  fake={lc[1]:,}')
    for s,c in sorted(sc.items())[:6]: print(f'    {s}: {c:,}')
    if len(sc) > 6: print(f'    ... and {len(sc)-6} more sources')

In [ ]:
# CelebDF — held-out test only
celeb_fake_paths, celeb_real_paths = [], []
for sn in ['Train','Val','Test']:
    sd = CELEB_ROOT / sn
    if not sd.exists(): continue
    for fn in ['fake','Fake']:
        fd = sd/fn
        if fd.exists(): celeb_fake_paths += list_images(fd); break
    for rn in ['real','Real']:
        rd = sd/rn
        if rd.exists(): celeb_real_paths += list_images(rd); break

random.shuffle(celeb_fake_paths); random.shuffle(celeb_real_paths)
celeb_test_refs  = [SampleRef(str(p),1,'celebdf_fake') for p in celeb_fake_paths[:15000]]
celeb_test_refs += [SampleRef(str(p),0,'celebdf_real') for p in celeb_real_paths[:8000]]
random.shuffle(celeb_test_refs)
cc = Counter(r.label for r in celeb_test_refs)
print(f'CelebDF held-out test: {len(celeb_test_refs):,}  real={cc[0]:,}  fake={cc[1]:,}')

In [ ]:
# ══════════════════════════════════════════════
#  LOGIT EXTRACTION DATASET
#
#  Strategy: pre-extract (freq_logit, spatial_logit) pairs
#  for all images, then train fusion head on those pairs.
#
#  WHY pre-extract:
#  Running both EfficientNet-B3 models per batch during fusion
#  training is expensive and slow on Kaggle.
#  Pre-extraction runs each model once over the dataset and saves
#  the logit pairs to tensors — fusion training then operates on
#  (2-dim input, label) pairs which is extremely fast.
#  ~220K images × 2 models × 10 epochs vs 220K × 2 models × 1 pass.
# ══════════════════════════════════════════════
class ImageDataset(Dataset):
    def __init__(self, refs, mode='frequency'):
        self.refs = refs
        self.mode = mode

    def __len__(self): return len(self.refs)

    def __getitem__(self, idx):
        r = self.refs[idx]
        try:
            img = Image.open(r.path).convert('RGB')
        except Exception:
            return torch.zeros(3, TARGET_SIZE, TARGET_SIZE), r.label, r.source
        if self.mode == 'frequency':
            return preprocess_frequency(img), r.label, r.source
        else:
            return preprocess_spatial(img), r.label, r.source


@torch.no_grad()
def extract_logits(model, refs, mode, desc):
    """Run model over all refs, return (N,) logit tensor."""
    ds = ImageDataset(refs, mode=mode)
    loader = DataLoader(ds, batch_size=128, shuffle=False,
                        num_workers=4, pin_memory=True)
    all_logits = []
    for x, _, _ in tqdm(loader, desc=desc):
        x = x.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE=='cuda')):
            logits = model(x).squeeze(1)
        all_logits.append(logits.cpu())
    return torch.cat(all_logits)  # (N,)


print('Logit extraction functions defined.')

In [ ]:
# ══════════════════════════════════════════════
#  EXTRACT LOGITS FROM BOTH BRANCHES
#  This is the one-time expensive step.
#  ~5-10 min per split per model on Kaggle T4.
# ══════════════════════════════════════════════
print('Extracting logits — this runs both models over all data once...')
print(f'Train: {len(train_refs):,}  Val: {len(val_refs):,}  Test: {len(test_refs):,}')
print(f'CelebDF test: {len(celeb_test_refs):,}  Phone test: {len(phone_test_refs):,}\n')

# Training set
print('--- TRAIN SET ---')
train_freq_logits    = extract_logits(freq_model,    train_refs, 'frequency', 'Train Freq')
train_spatial_logits = extract_logits(spatial_model, train_refs, 'spatial',   'Train Spat')
train_labels         = torch.tensor([r.label for r in train_refs], dtype=torch.float32)

# Validation set
print('\n--- VAL SET ---')
val_freq_logits    = extract_logits(freq_model,    val_refs, 'frequency', 'Val Freq')
val_spatial_logits = extract_logits(spatial_model, val_refs, 'spatial',   'Val Spat')
val_labels         = torch.tensor([r.label for r in val_refs], dtype=torch.float32)

# FF++ test set
print('\n--- FF++ TEST SET ---')
test_freq_logits    = extract_logits(freq_model,    test_refs, 'frequency', 'Test Freq')
test_spatial_logits = extract_logits(spatial_model, test_refs, 'spatial',   'Test Spat')
test_labels         = torch.tensor([r.label for r in test_refs], dtype=torch.float32)
test_sources        = [r.source for r in test_refs]

# CelebDF test set
print('\n--- CELEBDF TEST SET ---')
celeb_freq_logits    = extract_logits(freq_model,    celeb_test_refs, 'frequency', 'Celeb Freq')
celeb_spatial_logits = extract_logits(spatial_model, celeb_test_refs, 'spatial',   'Celeb Spat')
celeb_labels         = torch.tensor([r.label for r in celeb_test_refs], dtype=torch.float32)

# Phone test set
print('\n--- PHONE TEST SET ---')
phone_freq_logits    = extract_logits(freq_model,    phone_test_refs, 'frequency', 'Phone Freq')
phone_spatial_logits = extract_logits(spatial_model, phone_test_refs, 'spatial',   'Phone Spat')
phone_labels         = torch.tensor([r.label for r in phone_test_refs], dtype=torch.float32)

print('\nAll logits extracted.')
print(f'Train logits: freq={train_freq_logits.shape}  spat={train_spatial_logits.shape}')

# Show what each branch sees on average (diagnostic)
tr_fake_idx  = [i for i,r in enumerate(train_refs) if r.label==1]
tr_real_idx  = [i for i,r in enumerate(train_refs) if r.label==0]
print(f'\nBranch signals on training data:')
print(f'  Freq  logit real: {train_freq_logits[tr_real_idx].mean():.3f}  '
      f'fake: {train_freq_logits[tr_fake_idx].mean():.3f}')
print(f'  Spat  logit real: {train_spatial_logits[tr_real_idx].mean():.3f}  '
      f'fake: {train_spatial_logits[tr_fake_idx].mean():.3f}')

In [ ]:
# ══════════════════════════════════════════════
#  LOGIT PAIR DATASET — for fusion head training
#  Ultra-fast: each item is just 2 floats + 1 label
# ══════════════════════════════════════════════
class LogitPairDataset(Dataset):
    def __init__(self, freq_logits, spatial_logits, labels):
        self.freq    = freq_logits
        self.spatial = spatial_logits
        self.labels  = labels

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return (self.freq[idx], self.spatial[idx],
                self.labels[idx])


train_pair_ds = LogitPairDataset(train_freq_logits, train_spatial_logits, train_labels)
val_pair_ds   = LogitPairDataset(val_freq_logits,   val_spatial_logits,   val_labels)

train_pair_loader = DataLoader(train_pair_ds, batch_size=BATCH_SIZE,
                               shuffle=True,  num_workers=0)
val_pair_loader   = DataLoader(val_pair_ds,   batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=0)

print(f'LogitPairDataset ready.')
print(f'Train pairs: {len(train_pair_ds):,}  Val pairs: {len(val_pair_ds):,}')

# Class balance check
tc = Counter(train_labels.tolist())
ratio = tc[0.0] / max(tc[1.0], 1)
print(f'Train balance: real={tc[0.0]:,.0f}  fake={tc[1.0]:,.0f}  ratio={ratio:.2f}')
if ratio > 1.5 or ratio < 0.67:
    pos_w = torch.tensor([ratio], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    print(f'Imbalanced — pos_weight={ratio:.2f}')
else:
    criterion = nn.BCEWithLogitsLoss()
    print('Balanced — no pos_weight')

In [ ]:
optimizer = torch.optim.AdamW(fusion.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LR_MIN
)
print(f'AdamW lr={LR}  CosineAnnealing T_max={EPOCHS}')
print(f'Trainable fusion params: {sum(p.numel() for p in fusion.parameters() if p.requires_grad):,}')

In [ ]:
# ══════════════════════════════════════════════
#  EVALUATION ON PRE-EXTRACTED LOGITS
# ══════════════════════════════════════════════
THRESHOLD = 0.50  # will be tuned after training

@torch.no_grad()
def eval_logits(freq_logits, spatial_logits, labels, sources=None,
                threshold=THRESHOLD, desc='Eval'):
    fusion.eval()
    ds = LogitPairDataset(freq_logits, spatial_logits, labels)
    loader = DataLoader(ds, batch_size=1024, shuffle=False)
    all_probs, all_preds, all_labels = [], [], []

    for f_l, s_l, y in loader:
        f_l = f_l.to(DEVICE); s_l = s_l.to(DEVICE)
        final_logit = fusion(f_l, s_l)
        probs = torch.sigmoid(final_logit)
        preds = (probs >= threshold).long()
        all_probs.extend(probs.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.long().tolist())

    ov = {
        'acc':       accuracy_score(all_labels, all_preds),
        'macro_f1':  f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'real_f1':   f1_score(all_labels, all_preds, pos_label=0, zero_division=0),
        'fake_f1':   f1_score(all_labels, all_preds, pos_label=1, zero_division=0),
        'real_rec':  recall_score(all_labels, all_preds, pos_label=0, zero_division=0),
        'fake_rec':  recall_score(all_labels, all_preds, pos_label=1, zero_division=0),
        'mean_prob': float(np.mean(all_probs)),
    }
    try:
        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        ov['auc'] = float(auc(fpr, tpr))
        ov['fpr'] = fpr; ov['tpr'] = tpr
    except: ov['auc'] = 0.0; ov['fpr'] = []; ov['tpr'] = []

    # Per-source breakdown if sources provided
    per = {}
    if sources:
        for s in set(sources):
            idx = [i for i,ss in enumerate(sources) if ss==s]
            yt  = [all_labels[i] for i in idx]
            yp  = [all_preds[i]  for i in idx]
            pl  = 0 if yt[0]==0 else 1
            per[s] = {
                'acc': accuracy_score(yt,yp),
                'f1':  f1_score(yt,yp,pos_label=pl,zero_division=0),
                'rec': recall_score(yt,yp,pos_label=pl,zero_division=0),
                'n':   len(idx),
            }
    return ov, per, all_probs, all_labels


print(f'eval_logits() defined. Default threshold={THRESHOLD}')

In [ ]:
# ══════════════════════════════════════════════
#  FUSION HEAD TRAINING LOOP
#  Extremely fast — tiny MLP on pre-extracted logit pairs
#  Expected: ~30-60 seconds per epoch
# ══════════════════════════════════════════════
print(f'Training fusion head for {EPOCHS} epochs')
print(f'{len(train_pair_loader):,} batches/epoch  bs={BATCH_SIZE}')
print('='*55)

best_val_f1 = 0.0
history = []

for epoch in range(EPOCHS):
    fusion.train()
    run_loss = run_correct = run_n = 0

    for f_l, s_l, y in train_pair_loader:
        f_l = f_l.to(DEVICE); s_l = s_l.to(DEVICE); y = y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        final_logit = fusion(f_l, s_l)
        loss = criterion(final_logit, y)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            preds = (torch.sigmoid(final_logit) >= 0.5).long()
            run_correct += (preds == y.long()).sum().item()
        run_loss += loss.item() * f_l.size(0)
        run_n    += f_l.size(0)

    scheduler.step()
    tr_loss = run_loss / max(run_n,1)
    tr_acc  = run_correct / max(run_n,1)
    lr_now  = scheduler.get_last_lr()[0]

    fv, _, _, _ = eval_logits(val_freq_logits, val_spatial_logits, val_labels)

    print(f'Ep {epoch+1:02d}/{EPOCHS}'
          f'  loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}'
          f'  macro_f1={fv["macro_f1"]:.4f}'
          f'  real_f1={fv["real_f1"]:.4f}'
          f'  fake_f1={fv["fake_f1"]:.4f}'
          f'  mean_p={fv["mean_prob"]:.3f}'
          f'  lr={lr_now:.2e}')

    ckpt = {
        'fusion_state_dict': fusion.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch+1,
        'val_macro_f1': fv['macro_f1'],
    }
    torch.save(ckpt, f'{CKPT_DIR}/fusion_epoch_{epoch+1:02d}.pth')

    if fv['macro_f1'] > best_val_f1:
        best_val_f1 = fv['macro_f1']
        torch.save(ckpt, f'{CKPT_DIR}/best_fusion.pth')
        print(f'  >> best_fusion.pth  macro_f1={best_val_f1:.4f}')

    history.append({'ep':epoch+1,'loss':tr_loss,'tr_acc':tr_acc,
                    'macro_f1':fv['macro_f1'],'real_f1':fv['real_f1'],
                    'fake_f1':fv['fake_f1'],'mean_p':fv['mean_prob'],'lr':lr_now})

print('\nFusion head training complete.')

In [ ]:
print(f'{"Ep":>4}  {"Loss":>7}  {"TrAcc":>7}  {"MacroF1":>8}  '
      f'{"RealF1":>7}  {"FakeF1":>7}  {"MeanP":>6}  {"LR":>9}')
print('-'*70)
for h in history:
    print(f'{h["ep"]:>4}  {h["loss"]:>7.4f}  {h["tr_acc"]:>7.4f}  '
          f'{h["macro_f1"]:>8.4f}  {h["real_f1"]:>7.4f}  '
          f'{h["fake_f1"]:>7.4f}  {h["mean_p"]:>6.3f}  {h["lr"]:>9.2e}')

In [ ]:
best_ckpt = torch.load(f'{CKPT_DIR}/best_fusion.pth', map_location=DEVICE)
fusion.load_state_dict(best_ckpt['fusion_state_dict'])
print(f'Loaded best fusion head from epoch {best_ckpt["epoch"]}')
print(f'Val macro_f1: {best_ckpt["val_macro_f1"]:.4f}')

In [ ]:
# ══════════════════════════════════════════════
#  FULL EVALUATION — ALL TEST SETS
# ══════════════════════════════════════════════

print('FF++ Test...')
ff_ov, ff_per, ff_probs, ff_labels = eval_logits(
    test_freq_logits, test_spatial_logits, test_labels,
    sources=test_sources
)
print(f'  acc={ff_ov["acc"]:.4f}  macro_f1={ff_ov["macro_f1"]:.4f}  '
      f'real_f1={ff_ov["real_f1"]:.4f}  fake_f1={ff_ov["fake_f1"]:.4f}  '
      f'AUC={ff_ov["auc"]:.4f}')
print('  Per manipulation type:')
for s,m in sorted(ff_per.items(), key=lambda x:x[1]['f1']):
    print(f'    {s:35s}  f1={m["f1"]:.3f}  rec={m["rec"]:.3f}  n={m["n"]}')

print('\nCelebDF Test (cross-dataset, never seen in training)...')
ce_ov, ce_per, ce_probs, ce_labels = eval_logits(
    celeb_freq_logits, celeb_spatial_logits, celeb_labels
)
fpr_c, tpr_c, _ = roc_curve(ce_labels, ce_probs)
celeb_auc = auc(fpr_c, tpr_c)
print(f'  acc={ce_ov["acc"]:.4f}  macro_f1={ce_ov["macro_f1"]:.4f}  '
      f'real_f1={ce_ov["real_f1"]:.4f}  fake_f1={ce_ov["fake_f1"]:.4f}')
print(f'  fake_recall={ce_ov["fake_rec"]:.4f}  real_recall={ce_ov["real_rec"]:.4f}')
print(f'  CelebDF AUC: {celeb_auc:.4f}  (Published SOTA: 65-76%)')

print('\nPhone Images Test (all real, held-out)...')
ph_ov, _, ph_probs, ph_labels = eval_logits(
    phone_freq_logits, phone_spatial_logits, phone_labels
)
print(f'  Total: {len(phone_test_refs):,} real phone images')
print(f'  Called REAL: {int(ph_ov["real_rec"]*len(phone_test_refs))}/{len(phone_test_refs)}'
      f'  ({ph_ov["real_rec"]*100:.1f}%)')
print(f'  mean_prob: {ph_ov["mean_prob"]:.4f}')

In [ ]:
# ══════════════════════════════════════════════
#  PARETO THRESHOLD ON VALIDATION SET
# ══════════════════════════════════════════════
_, _, val_probs_full, val_labels_full = eval_logits(
    val_freq_logits, val_spatial_logits, val_labels
)

fpr_v, tpr_v, roc_thr = roc_curve(val_labels_full, val_probs_full)
val_auc = auc(fpr_v, tpr_v)

sweep = np.linspace(0.05, 0.95, 500)
FAKE_TARGET = 0.93; REAL_TARGET = 0.88
results = []
for thr in sweep:
    p       = [1 if pp>=thr else 0 for pp in val_probs_full]
    rf1     = f1_score(val_labels_full, p, pos_label=0, zero_division=0)
    ff1     = f1_score(val_labels_full, p, pos_label=1, zero_division=0)
    macro   = (rf1+ff1)/2
    penalty = max(0, FAKE_TARGET-ff1) + max(0, REAL_TARGET-rf1)
    results.append({'thr':float(thr),'real_f1':rf1,'fake_f1':ff1,'macro':macro,'penalty':penalty})

best = min(results, key=lambda x:(x['penalty'],-x['macro']))
pareto_thr = best['thr']

print(f'Val AUC: {val_auc:.4f}')
print(f'Pareto threshold: {pareto_thr:.3f}')
print(f'  real_f1={best["real_f1"]:.4f}  fake_f1={best["fake_f1"]:.4f}  '
      f'macro={best["macro"]:.4f}  penalty={best["penalty"]:.4f}')
print(f'  Status: {"BOTH TARGETS MET" if best["penalty"]==0 else "BEST AVAILABLE"}')
print(f'\n>>> FUSED SYSTEM THRESHOLD = {pareto_thr:.3f} <<<')

In [ ]:
# Re-evaluate all test sets at Pareto threshold
print(f'\n=== Results at Pareto threshold={pareto_thr:.3f} ===')

for name, f_l, s_l, lbls, srcs in [
    ('FF++ Test',         test_freq_logits,  test_spatial_logits,  test_labels,  test_sources),
    ('CelebDF',           celeb_freq_logits, celeb_spatial_logits, celeb_labels, None),
    ('Phone images',      phone_freq_logits, phone_spatial_logits, phone_labels, None),
]:
    ov, per, probs, labels = eval_logits(f_l, s_l, lbls, srcs, threshold=pareto_thr)
    try:
        fpr_, tpr_, _ = roc_curve(labels, probs)
        auc_val = auc(fpr_, tpr_)
    except: auc_val = 0.0
    print(f'\n{name}:')
    print(f'  acc={ov["acc"]:.4f}  macro_f1={ov["macro_f1"]:.4f}  '
          f'real_f1={ov["real_f1"]:.4f}  fake_f1={ov["fake_f1"]:.4f}')
    print(f'  real_recall={ov["real_rec"]:.4f}  fake_recall={ov["fake_rec"]:.4f}  '
          f'AUC={auc_val:.4f}  mean_p={ov["mean_prob"]:.4f}')
    if per:
        for s,m in sorted(per.items(), key=lambda x:x[1]['f1']):
            print(f'    {s:35s}  f1={m["f1"]:.3f}  rec={m["rec"]:.3f}  n={m["n"]}')

In [ ]:
# ══════════════════════════════════════════════
#  COMPARISON: Individual branches vs Fused
# ══════════════════════════════════════════════
print('\n' + '='*70)
print('COMPARISON: Individual branches vs Fused system')
print('='*70)
print(f'{"Metric":35s}  {"FreqOnly":>10}  {"SpatOnly":>10}  {"FUSED":>10}')
print('-'*70)

# Previously measured branch-only numbers
rows = [
    ('CelebDF AUC',             0.5123, 0.7255, celeb_auc),
    ('CelebDF fake_recall@0.5', 0.3575, 0.3229, ce_ov['fake_rec']),
    ('Phone real_recall',       0.2700, 0.8858, ph_ov['real_rec']),
    ('FF++ macro_f1',           0.8988, 0.9359, ff_ov['macro_f1']),
    ('FF++ real_f1',            0.8231, 0.8978, ff_ov['real_f1']),
    ('FF++ fake_f1',            0.9602, 0.9741, ff_ov['fake_f1']),
]
for name, f_val, s_val, fused_val in rows:
    best_mark = '>>>' if fused_val >= max(f_val, s_val) else '   '
    print(f'{name:35s}  {f_val:>10.4f}  {s_val:>10.4f}  {best_mark}{fused_val:>7.4f}')

# ROC curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(fpr_c, tpr_c, '#e74c3c', lw=2,
        label=f'Fused AUC={celeb_auc:.4f}')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
ax.axhspan(0.65, 0.76, alpha=0.1, color='green', label='Published SOTA range')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('CelebDF ROC\n(cross-dataset, never trained on)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
thrs = [r['thr']     for r in results]
mf1s = [r['macro']   for r in results]
rf1s = [r['real_f1'] for r in results]
ff1s = [r['fake_f1'] for r in results]
ax.plot(thrs, mf1s, '#9b59b6', lw=2.5, label='Macro F1')
ax.plot(thrs, rf1s, '#27ae60', lw=1.5, linestyle='--', label='Real F1')
ax.plot(thrs, ff1s, '#e74c3c', lw=1.5, linestyle='--', label='Fake F1')
ax.axvline(pareto_thr, '#9b59b6', lw=2, linestyle=':', label=f'Pareto={pareto_thr:.3f}')
ax.axhline(REAL_TARGET, color='#27ae60', lw=1, linestyle=':', alpha=0.5)
ax.axhline(FAKE_TARGET, color='#e74c3c', lw=1, linestyle=':', alpha=0.5)
ax.set_xlabel('Threshold'); ax.set_ylabel('F1')
ax.set_title('Fused System F1 vs Threshold', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[2]
rp = [val_probs_full[i] for i,l in enumerate(val_labels_full) if l==0]
fp = [val_probs_full[i] for i,l in enumerate(val_labels_full) if l==1]
ax.hist(rp, bins=60, alpha=0.6, color='#27ae60',
        label=f'Real (n={len(rp):,})', density=True)
ax.hist(fp, bins=60, alpha=0.6, color='#e74c3c',
        label=f'Fake (n={len(fp):,})', density=True)
ax.axvline(pareto_thr, '#9b59b6', lw=2, linestyle='--', label=f'Pareto={pareto_thr:.3f}')
ax.set_xlabel('Fake Probability'); ax.set_ylabel('Density')
ax.set_title('Fused Probability Distribution\n(good = peaks near 0 and 1)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(f'DefakeX Fused System | Val AUC={val_auc:.4f}  '
             f'CelebDF AUC={celeb_auc:.4f}  Pareto thr={pareto_thr:.3f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/fusion_analysis.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: fusion_analysis.png')

In [ ]:
print('\n' + '='*65)
print('DEFAKEX FUSED SYSTEM — FINAL RESULTS')
print('='*65)
print(f'Frequency branch:  epoch {freq_ckpt.get("epoch","?")}  (frozen)')
print(f'Spatial branch v2: epoch {sp_ckpt.get("epoch","?")}  (frozen)')
print(f'Fusion head:       epoch {best_ckpt["epoch"]}')
print(f'Pareto threshold:  {pareto_thr:.3f}')

for name, f_l, s_l, lbls in [
    ('FF++ Test',   test_freq_logits,  test_spatial_logits,  test_labels),
    ('CelebDF',     celeb_freq_logits, celeb_spatial_logits, celeb_labels),
    ('Phone Real',  phone_freq_logits, phone_spatial_logits, phone_labels),
]:
    ov, _, probs, labels = eval_logits(f_l, s_l, lbls, threshold=pareto_thr)
    try:
        fpr_, tpr_, _ = roc_curve(labels, probs)
        auc_val = f'{auc(fpr_, tpr_):.4f}'
    except: auc_val = 'N/A'
    print(f'\n{name}:')
    print(f'  macro_f1={ov["macro_f1"]:.4f}  real_f1={ov["real_f1"]:.4f}  '
          f'fake_f1={ov["fake_f1"]:.4f}  AUC={auc_val}')
    print(f'  real_recall={ov["real_rec"]:.4f}  fake_recall={ov["fake_rec"]:.4f}')

print('\nFusion head learned weights (diagnostic):')
w1 = fusion.net[0].weight.data.cpu().numpy()
print(f'  Layer 1 weights [freq, spatial]: {w1.mean(axis=0).round(3)}')
print(f'  Positive = branch signal increases fake probability')
print(f'  Magnitude = how much the head trusts each branch')

In [ ]:
from IPython.display import FileLink
import os
print([f for f in os.listdir(CKPT_DIR) if 'fusion' in f])
FileLink('best_fusion.pth')